In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [4]:
# ==========================================
#  Import Libraries
# ==========================================
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# ==========================================
#  Load Data
# ==========================================
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

# ==========================================
#  Handle Missing Values (TRAIN)
# ==========================================
train["Age"] = train["Age"].fillna(train["Age"].mean())
train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])

# ==========================================
# Feature Engineering (TRAIN)
# ==========================================
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
train["IsAlone"] = 0
train.loc[train["FamilySize"] == 1, "IsAlone"] = 1

# ==========================================
#  Convert Categorical (TRAIN)
# ==========================================
train["Sex"] = train["Sex"].map({"male": 0, "female": 1})
train = pd.get_dummies(train, columns=["Embarked"], drop_first=True)

# ==========================================
#  Drop Unnecessary Columns (TRAIN)
# ==========================================
train = train.drop(
    ["PassengerId", "Name", "Ticket", "Cabin", "SibSp", "Parch"],
    axis=1,
    errors="ignore"
)

# ==========================================
#  Separate Features & Target
# ==========================================
y = train["Survived"]
X = train.drop("Survived", axis=1)

# ==========================================
#  Train Random Forest on FULL DATA
# ==========================================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X, y)

# ==========================================
#  Preprocess TEST Data (Same Steps)
# ==========================================
test["Age"] = test["Age"].fillna(train["Age"].mean())
test["Fare"] = test["Fare"].fillna(train["Fare"].mean())

# Create features
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1
test["IsAlone"] = 0
test.loc[test["FamilySize"] == 1, "IsAlone"] = 1

# Convert categorical
test["Sex"] = test["Sex"].map({"male": 0, "female": 1})
test = pd.get_dummies(test, columns=["Embarked"], drop_first=True)

# Drop unused columns
test = test.drop(
    ["Name", "Ticket", "Cabin", "SibSp", "Parch"],
    axis=1,
    errors="ignore"
)

# ==========================================
# Align Test Columns With Train Columns
# ==========================================
test = test.reindex(columns=X.columns, fill_value=0)

# ==========================================
#  Predict
# ==========================================
predictions = model.predict(test)

# ==========================================
# Create Submission File
# ==========================================
submission = pd.DataFrame({
    "PassengerId": pd.read_csv("/kaggle/input/competitions/titanic/test.csv")["PassengerId"],
    "Survived": predictions
})

submission.to_csv("submission.csv", index=False)

print("Submission file created successfully!")

Submission file created successfully!
